In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
import warnings
warnings.filterwarnings("ignore", message=".*langchain-community.*")

In [3]:
GOOGLE_API_KEY=os.getenv("GOOGLE_API_KEY")
TAVILY_API_KEY=os.getenv("TAVILY_API_KEY")
GROQ_API_KEY=os.getenv("GROQ_API_KEY")
LANGSMITH_API_KEY=os.getenv("LANGSMITH_API_KEY")
LANGSMITH_PROJECT=os.getenv("LANGSMITH_PROJECT")

In [4]:
os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY
os.environ["TAVILY_API_KEY"] = TAVILY_API_KEY
os.environ["GROQ_API_KEY"]= GROQ_API_KEY
os.environ["LANGSMITH_API_KEY"] = LANGSMITH_API_KEY
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_ENDPOINT"] = "https://eu.api.smith.langchain.com"
os.environ["LANGSMITH_PROJECT"]=LANGSMITH_PROJECT

In [ ]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")

In [7]:
from langchain_groq import ChatGroq
llm = ChatGroq(model="llama-3.3-70b-versatile")

In [8]:
llm.invoke("hi,how are you?")

AIMessage(content="Hello, I'm doing well, thanks for asking. I'm a large language model, so I don't have feelings or emotions like humans do, but I'm always happy to help and chat with you. How about you? How's your day going so far?", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 55, 'prompt_tokens': 41, 'total_tokens': 96, 'completion_time': 0.119148551, 'completion_tokens_details': None, 'prompt_time': 0.00220808, 'prompt_tokens_details': None, 'queue_time': 0.093683119, 'total_time': 0.121356631}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_f8b414701e', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019ef593-9e5b-7a70-aa3d-2c9b424af7a7-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 41, 'output_tokens': 55, 'total_tokens': 96})

# Predefine tools

In [12]:
import wikipedia

wikipedia.set_user_agent("Mozilla/5.0 (Windows NT 10.0; Win64; x64) ")

In [13]:
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper
api_wrapper = WikipediaAPIWrapper()
tool = WikipediaQueryRun(api_wrapper=api_wrapper)
print(tool.run({"query": "langchain"}))

JSONDecodeError: Expecting value: line 1 column 1 (char 0)

In [14]:
from langchain_community.tools import YouTubeSearchTool
tool2=YouTubeSearchTool()
tool2.run("sunny savita")

"['https://www.youtube.com/watch?v=Tf2ZzrCBJUI&pp=ygUMc3Vubnkgc2F2aXRh', 'https://www.youtube.com/watch?v=_kpxLkH5vY0&pp=ygUMc3Vubnkgc2F2aXRh']"

In [15]:
from langchain_community.tools.tavily_search import TavilySearchResults
tool = TavilySearchResults()
tool.invoke({"query": "who is mohandas karamchand gandhi?"})

C:\Users\pragi\AppData\Local\Temp\ipykernel_24628\911929189.py:2: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the `langchain-tavily package and should be used instead. To use it run `pip install -U `langchain-tavily` and import as `from `langchain_tavily import TavilySearch``.
  tool = TavilySearchResults()


[{'title': 'Mohandas Gandhi | Accomplishments, Beliefs & Profession - Lesson | Study.com',
  'url': 'https://study.com/learn/lesson/mohandas-gandhi.html',
  'content': "##### French Revolution Study Guide\n\n##### World History I\n\n##### SAT Subject Test World History: Practice and Study Guide\n\n## Who was Mohandas Gandhi?\n\nMahatma Gandhi's full name is Mohandas Karamchand Gandhi. He was born on October 2, 1869, in Porbandar, India. Gandhi's professions included being a lawyer, social activist, politician, and writer. Gandhi was famous for his role as an activist leader for the movement against the British rule of India. This movement followed a peaceful doctrine and used nonviolent protest as a means to achieve change. [...] Mahatma Gandhi is also known by his full name Mohandas Karamchand Gandhi. Mohandas Gandhi's father, Karamchand Gandhi, was the chief minister of their hometown in Porbandar, India, and Mohandas Gandhi's mother, Putlibai Gandhi, was deeply religious. Gandhi was

# Create a Custom tool

In [17]:
from langchain_classic.agents import tool
@tool
def get_word_length(word: str) -> int:
    """Returns the length of a word."""
    return len(word)

In [18]:
get_word_length.invoke("abc")

3

In [19]:
@tool
def multiply(a: int, b: int) -> int:
    """Multiply two numbers."""
    return a * b

In [20]:
@tool
def sum(a: int, b: int) -> int:
    """Adding two numbers."""
    return a + b

In [21]:
print(sum.name)
print(sum.description)
print(sum.args)

sum
Adding two numbers.
{'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}


In [22]:
multiply.invoke({"a":10,"b":20})

200

### Concept of Agents

In [29]:
# legacy agent imports - these still work
from langchain_classic.agents import AgentType, initialize_agent

# load_tools is broken (needs mypy_extensions + langchain_community)
# we manually create the tool instead - same result
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper

tool = [WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper())]

In [30]:
agent = initialize_agent(tool, llm, agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION, verbose=True)

C:\Users\pragi\AppData\Local\Temp\ipykernel_24628\2451433261.py:1: LangChainDeprecationWarning: Use `langchain.agents.create_agent` for new applications. It provides a more flexible agent factory with middleware support, structured output, and integration with LangGraph for persistence, streaming, and human-in-the-loop workflows. Migration guide: https://docs.langchain.com/oss/python/migrate/langchain-v1
  agent = initialize_agent(tool, llm, agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION, verbose=True)


In [31]:
agent.run("what is llama and who create this llm model?")

C:\Users\pragi\AppData\Local\Temp\ipykernel_24628\1212027048.py:1: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain-classic 0.1.0 and will be removed in 2.0.0. Use `invoke` instead.
  agent.run("what is llama and who create this llm model?")




> Entering new AgentExecutor chain...
Thought: To answer this question, I need to find information about the LLaMA model, which is a type of artificial intelligence language model. I should start by searching for LLaMA on Wikipedia to see what information is available.

Action: wikipedia
Action Input: LLaMA

JSONDecodeError: Expecting value: line 1 column 1 (char 0)

In [33]:
from langchain_community.tools.tavily_search import TavilySearchResults

search = TavilySearchResults()

In [34]:
search.invoke("what is the weather in SF")

[{'title': 'Weather San Francisco in June 2026: Temperature & Climate',
  'url': 'https://en.climate-data.org/north-america/united-states-of-america/california/san-francisco-385/t/june-6',
  'content': '| 23. June | 16 °C | 61 °F | 22 °C | 72 °F | 12 °C | 54 °F | 12 °C | 54 °F | 0.0 mm | 0.0 inch. |\n| 24. June | 16 °C | 61 °F | 22 °C | 71 °F | 12 °C | 54 °F | 12 °C | 54 °F | 0.1 mm | 0.0 inch. |\n| 25. June | 16 °C | 61 °F | 22 °C | 71 °F | 12 °C | 54 °F | 12 °C | 54 °F | 0.1 mm | 0.0 inch. |\n| 26. June | 16 °C | 61 °F | 22 °C | 72 °F | 12 °C | 54 °F | 13 °C | 55 °F | 0.1 mm | 0.0 inch. |\n| 27. June | 16 °C | 61 °F | 22 °C | 71 °F | 12 °C | 54 °F | 13 °C | 55 °F | 0.1 mm | 0.0 inch. |\n| 28. June | 16 °C | 62 °F | 22 °C | 71 °F | 13 °C | 55 °F | 13 °C | 55 °F | 0.2 mm | 0.0 inch. |\n| 29. June | 16 °C | 62 °F | 22 °C | 72 °F | 12 °C | 54 °F | 13 °C | 55 °F | 0.1 mm | 0.0 inch. |\n| 30. June | 17 °C | 62 °F | 22 °C | 72 °F | 13 °C | 55 °F | 13 °C | 55 °F | 0.1 mm | 0.0 inch. | [...] 

In [35]:
tools = [search]

In [36]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant"),
    MessagesPlaceholder(variable_name="chat_history", optional=True),
    ("human", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad"),
])

In [37]:
from langchain_classic.agents import create_tool_calling_agent
agent = create_tool_calling_agent(llm, tools, prompt)

In [38]:
from langchain_classic.agents import AgentExecutor
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

In [39]:
agent_executor.invoke({"input": "hello how are you?"})



> Entering new AgentExecutor chain...
I'm doing well, thanks for asking. Is there something I can help you with or would you like to chat?

> Finished chain.


{'input': 'hello how are you?',
 'output': "I'm doing well, thanks for asking. Is there something I can help you with or would you like to chat?"}

In [40]:
agent_executor.invoke({"input": "whats the weather in sf?"})



> Entering new AgentExecutor chain...

Invoking: `tavily_search_results_json` with `{'query': 'San Francisco weather today'}`


[{'title': 'San Francisco Weather Conditions: Temperature | 30 Days Forecast', 'url': 'https://www.aqi.in/weather/us/united-states/california/san-francisco', 'content': "From 07:15 AM 23 June 2026, San Francisco's 10-day forecast shows this trend:\nToday (Jun. 23) : Temp. 14°C, Hum. 92% and Fog condition.\nWednesday (Jun. 24) : Temp. 15°C, Hum. 78% and Sunny condition.\nThursday (Jun. 25) : Temp. 15°C, Hum. 79% and Sunny condition.\nFriday (Jun. 26) : Temp. 16°C, Hum. 84% and Partly Cloudy condition.\nSaturday (Jun. 27) : Temp. 16°C, Hum. 71% and Sunny condition.\nSunday (Jun. 28) : Temp. 16°C, Hum. 70% and Sunny condition.\nMonday (Jun. 29) : Temp. 16°C, Hum. 74% and Sunny condition.\nTuesday (Jun. 30) : Temp. 15°C, Hum. 74% and Sunny condition.\nWednesday (Jul. 1) : Temp. 17°C, Hum. 73% and Sunny condition.\nThursday (Jul. 2) : Temp. 17°C, Hum. 74% and Pat

{'input': 'whats the weather in sf?',
 'output': 'The current weather in San Francisco is mostly cloudy with a temperature of 14°C (57.2°F) and a humidity of 87%. The 10-day forecast shows a mix of sunny and cloudy days with temperatures ranging from 12°C to 19°C (54°F to 66°F). There is a 35% chance of rainfall in San Francisco later today.'}

### 1. RAG tool

In [41]:
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter

USER_AGENT environment variable not set, consider setting it to identify your requests.


In [42]:
loader = WebBaseLoader("https://docs.smith.langchain.com/overview")
docs = loader.load()

In [43]:
documents = RecursiveCharacterTextSplitter(
    chunk_size=1000, chunk_overlap=200
).split_documents(docs)

In [44]:
vector = FAISS.from_documents(documents,embeddings)
retriever = vector.as_retriever()

In [45]:
retriever.invoke("how to upload a dataset")[0]

Document(id='517dba83-8a9b-45bf-8e49-45258cbb1003', metadata={'source': 'https://docs.smith.langchain.com/overview', 'title': 'LangSmith Observability - Docs by LangChain', 'description': 'Instrument your LLM application, investigate traces, and monitor performance in production with LangSmith.', 'language': 'en'}, page_content='Connect these docs to Claude, VSCode, and more via MCP for real-time answers.Edit this page on GitHub or file an issue.⌘I')

In [48]:
from langchain_classic.tools.retriever import create_retriever_tool
retriever_tool = create_retriever_tool(
    retriever,
    "langsmith_search",
    "Search for information about LangSmith. For any questions about LangSmith, you must use this tool!",
)

In [49]:
tools = [search, retriever_tool]

In [51]:
from langchain_classic.agents import create_tool_calling_agent

agent = create_tool_calling_agent(llm, tools, prompt)

In [52]:
from langchain_classic.agents import AgentExecutor

agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

In [53]:
agent_executor.invoke({"input": "hi! what is a langsmith?"})



> Entering new AgentExecutor chain...

Invoking: `langsmith_search` with `{'query': 'what is LangSmith'}`


cause, and resolve them with LangSmith Engine.For terminology and core concepts, refer to Observability concepts. For trace pricing, retention, and limits, see Usage and billing.To set up a LangSmith instance, visit the Platform setup section to choose between cloud, hybrid, or self-hosted. All options include observability, evaluation, prompt engineering, and deployment.

LangSmith Observability - Docs by LangChainDocumentation IndexFetch the complete documentation index at: /llms.txtUse this file to discover all available pages before exploring further.Skip to main contentDocs by LangChain home pageMonitorSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationLangSmith ObservabilityOverviewEngineTraceDebugObserveLangSmith ObservabilityLangSmith Observability provides full visibility into your LLM application: from individual traces to production-wide performance 

{'input': 'hi! what is a langsmith?',
 'output': 'A LangSmith is an observability platform that provides full visibility into your LLM (Large Language Model) application. It offers features such as tracing, debugging, and monitoring to help you understand and improve the performance of your LLM application. LangSmith works with many frameworks and providers, including OpenAI, Anthropic, and Vercel AI SDK, and provides tools for automating workflows, collecting feedback, and detecting and resolving issues.'}

In [54]:
agent_executor.invoke({"input": "whats the weather in sf?"})



> Entering new AgentExecutor chain...

Invoking: `tavily_search_results_json` with `{'query': 'San Francisco weather today'}`


[{'title': 'San Francisco Weather Conditions: Temperature | 30 Days Forecast', 'url': 'https://www.aqi.in/weather/in/united-states/california/san-francisco', 'content': "From 06:15 AM 23 June 2026, San Francisco's 10-day forecast shows this trend:\nToday (Jun. 23) : Temp. 15°C, Hum. 88% and Fog condition.\nWednesday (Jun. 24) : Temp. 16°C, Hum. 78% and Sunny condition.\nThursday (Jun. 25) : Temp. 15°C, Hum. 78% and Sunny condition.\nFriday (Jun. 26) : Temp. 16°C, Hum. 82% and Sunny condition.\nSaturday (Jun. 27) : Temp. 15°C, Hum. 76% and Sunny condition.\nSunday (Jun. 28) : Temp. 17°C, Hum. 64% and Sunny condition.\nMonday (Jun. 29) : Temp. 17°C, Hum. 69% and Sunny condition.\nTuesday (Jun. 30) : Temp. 18°C, Hum. 65% and Sunny condition.\nWednesday (Jul. 1) : Temp. 17°C, Hum. 73% and Sunny condition.\nThursday (Jul. 2) : Temp. 16°C, Hum. 79% and Patchy rain

{'input': 'whats the weather in sf?',
 'output': 'The current weather in San Francisco is mostly cloudy with a high of 68F and a low of 54F. The weather forecast for the next 10 days shows a mix of sunny and cloudy days with temperatures ranging from the mid-60s to the mid-70s. There is a 35% chance of rainfall in San Francisco later today.'}

#### Add memory component

In [59]:
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_community.chat_message_histories import ChatMessageHistory

In [60]:
message_history=ChatMessageHistory()

In [61]:
agent_with_chat_history = RunnableWithMessageHistory(
    agent_executor,
    # This is needed because in most real world scenarios, a session id is needed
    # It isn't really used here because we are using a simple in memory ChatMessageHistory
    lambda session_id: message_history,
    input_messages_key="input",
    history_messages_key="chat_history",
)

d:\AI-ML\vscode\projects_dir\LangGraph_Shakedown\.venv\Lib\site-packages\IPython\core\interactiveshell.py:3748: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [62]:
agent_with_chat_history.invoke(
    {"input": "hi! my name is pragi how are you?"},
    config={"configurable": {"session_id": "firstchat"}},
)



> Entering new AgentExecutor chain...
I'm doing well, thank you for asking. It's nice to meet you, Pragi. Is there something I can help you with or would you like to chat?

> Finished chain.


{'input': 'hi! my name is pragi how are you?',
 'chat_history': [],
 'output': "I'm doing well, thank you for asking. It's nice to meet you, Pragi. Is there something I can help you with or would you like to chat?"}

### 2. ReAct

In [64]:
from langchain_classic.utilities import GoogleSerperAPIWrapper
from langchain_classic.agents import Tool
from langchain_core.prompts import PromptTemplate
from langchain_classic.agents import AgentExecutor, create_react_agent

In [65]:
google_search = GoogleSerperAPIWrapper()
tools = [
    Tool(
        name="Intermediate Answer",
        func=google_search.run,
        description="useful for when you need to ask with search",
        verbose=True
    )
]

In [67]:
template = '''Answer the following questions as best you can. You have access to the following tools:
{tools}
Use the following format:
Question: the input question you must answer 
Thought: you should always think about what to do
Action: the action to take, should be one of [{tool_names}]
Action Input: the input to the action
Observation: the result of the action
... (this Thought/Action/Action Input/Observation can repeat N times)
Thought: I now know the final answer
Final Answer: the final answer to the original input question
Begin!
Question: {input}
Thought:{agent_scratchpad}'''

In [68]:
prompt = PromptTemplate.from_template(template)

In [69]:
search_agent = create_react_agent(llm,tools,prompt)

In [70]:
agent_executor = AgentExecutor(
    agent=search_agent,
    tools=tools,
    verbose=True,
    return_intermediate_steps=True,
)

In [71]:
agent_executor.invoke({"input": "Where is the hometown of the 2007 US PGA championship winner and his score?"})



> Entering new AgentExecutor chain...
Thought: To answer this question, I need to find out who the 2007 US PGA championship winner was, his hometown, and his score. I can start by searching for the winner of the 2007 US PGA championship.

Action: Intermediate Answer
Action Input: 2007 US PGA championship winner POS PLAYER TOT 1 T. Woods 272 2 W. Austin 274 3 E. Els 275 T4 A. Oberholser 279The winner of the 2007 US PGA championship was Tiger Woods, with a score of 272. However, I still need to find out his hometown.

Action: Intermediate Answer
Action Input: Tiger Woods hometownHe grew up in Cypress, California. He took an interest in golf at age 6 months, watching as his father hit golf balls into a net and imitating his swing. He appeared on The Mike Douglas Show at age 2, putting with Bob Hope.Thought: I now know the final answer
Final Answer: The hometown of the 2007 US PGA championship winner, Tiger Woods, is Cypress, California, and his score was 272.

> Finished chain.


{'input': 'Where is the hometown of the 2007 US PGA championship winner and his score?',
 'output': 'The hometown of the 2007 US PGA championship winner, Tiger Woods, is Cypress, California, and his score was 272.',
 'intermediate_steps': [(AgentAction(tool='Intermediate Answer', tool_input='2007 US PGA championship winner', log='Thought: To answer this question, I need to find out who the 2007 US PGA championship winner was, his hometown, and his score. I can start by searching for the winner of the 2007 US PGA championship.\n\nAction: Intermediate Answer\nAction Input: 2007 US PGA championship winner'),
   ' POS PLAYER TOT 1 T. Woods 272 2 W. Austin 274 3 E. Els 275 T4 A. Oberholser 279'),
  (AgentAction(tool='Intermediate Answer', tool_input='Tiger Woods hometown', log='The winner of the 2007 US PGA championship was Tiger Woods, with a score of 272. However, I still need to find out his hometown.\n\nAction: Intermediate Answer\nAction Input: Tiger Woods hometown'),
   'He grew up in

### ReAct Agent with Custom tools

In [73]:

from langchain.tools import tool

# Custom tool for the Agent
@tool
def get_employee_id(name):
  """
  To get employee id, it takes employee name as arguments
  name(str): Name of the employee
  """
  fake_employees = {
    "Alice": "E001",
    "Bob": "E002",
    "Charlie": "E003",
    "Diana": "E004",
    "Evan": "E005",
    "Fiona": "E006",
    "George": "E007",
    "Hannah": "E008",
    "Ian": "E009",
    "Jasmine": "E010"}

  return fake_employees.get(name,"Employee not found")

In [74]:
# Custom tool for the Agent
@tool
def get_employee_salary(employee_id):
  """
  To get the salary of an employee, it takes employee_id as input and return salary
  """
  employee_salaries = {
    "E001": 56000,
    "E002": 47000,
    "E003": 52000,
    "E004": 61000,
    "E005": 45000,
    "E006": 58000,
    "E007": 49000,
    "E008": 53000,
    "E009": 50000,
    "E010": 55000
    }
  return employee_salaries.get(employee_id,"Employee not found")

In [75]:
print(prompt.template)

Answer the following questions as best you can. You have access to the following tools:
{tools}
Use the following format:
Question: the input question you must answer 
Thought: you should always think about what to do
Action: the action to take, should be one of [{tool_names}]
Action Input: the input to the action
Observation: the result of the action
... (this Thought/Action/Action Input/Observation can repeat N times)
Thought: I now know the final answer
Final Answer: the final answer to the original input question
Begin!
Question: {input}
Thought:{agent_scratchpad}


In [76]:
tools = [get_employee_salary, get_employee_id]

In [77]:
agent = create_react_agent(llm,tools,prompt)

In [78]:
agent_executor = AgentExecutor(agent=agent,tools=tools,verbose=True)

In [79]:
agent_executor.invoke({"input":"What is the Salary of Evan?"})



> Entering new AgentExecutor chain...
Thought: To find the salary of Evan, I first need to get Evan's employee ID, since the function to get the salary requires the employee ID as input.

Action: get_employee_id
Action Input: EvanE005Now that I have Evan's employee ID, I can use it to get his salary.

Action: get_employee_salary
Action Input: E00545000I now know the final answer
Final Answer: 45000

> Finished chain.


{'input': 'What is the Salary of Evan?', 'output': '45000'}

### 3. agent code from latest versions

In [84]:
# Import relevant functionality
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import MemorySaver
from langchain.agents import create_agent

In [85]:

# Create the agent
memory = MemorySaver()
search = TavilySearchResults(max_results=2)
tools = [search]
agent_executor = create_agent(llm, tools, checkpointer=memory)


In [86]:

# Use the agent
config = {"configurable": {"thread_id": "abc123"}}
for chunk in agent_executor.stream(
    {"messages": [HumanMessage(content="hi im bob! and i live in sf")]}, config
):
    print(chunk)
    print("----")

{'model': {'messages': [AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'tk413cjxq', 'function': {'arguments': '{"query":"San Francisco current events"}', 'name': 'tavily_search_results_json'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 21, 'prompt_tokens': 287, 'total_tokens': 308, 'completion_time': 0.064222815, 'completion_tokens_details': None, 'prompt_time': 0.033352747, 'prompt_tokens_details': None, 'queue_time': 0.348794671, 'total_time': 0.097575562}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_45180df409', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019ef5f2-65b6-73d3-b61d-0cb27a9c88ec-0', tool_calls=[{'name': 'tavily_search_results_json', 'args': {'query': 'San Francisco current events'}, 'id': 'tk413cjxq', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 287, 'output_tokens': 21, 'total_tokens': 308})

In [87]:

for chunk in agent_executor.stream(
    {"messages": [HumanMessage(content="whats the weather where I live?")]}, config
):
    print(chunk)
    print("----")

{'model': {'messages': [AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'h9e1hqp79', 'function': {'arguments': '{"query":"San Francisco weather"}', 'name': 'tavily_search_results_json'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 20, 'prompt_tokens': 1571, 'total_tokens': 1591, 'completion_time': 0.030339282, 'completion_tokens_details': None, 'prompt_time': 0.302012147, 'prompt_tokens_details': None, 'queue_time': 0.041513005, 'total_time': 0.332351429}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_f8b414701e', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019ef5f2-7a8c-7ba2-a450-729241ce8f9d-0', tool_calls=[{'name': 'tavily_search_results_json', 'args': {'query': 'San Francisco weather'}, 'id': 'h9e1hqp79', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 1571, 'output_tokens': 20, 'total_tokens': 1591})]}}
----
{